# Disney AI: Dynamic Collaboration — Advanced Notebook

This notebook implements a small game-theory engine inspired by Fantasia (1940).

**What it does**
- Approximates a mixed-strategy *minimax* solution on a 4x4 payoff matrix (Apprentice vs Conductor).
- Simulates 200 steps of repeated interaction with adaptive probabilities.
- Saves two charts to `img/` and prints a metrics dictionary you can quote in the README.

In [ ]:
# Setup
import os, json, numpy as np, matplotlib.pyplot as plt
np.random.seed(1)
os.makedirs("img", exist_ok=True)
plt.rcParams["figure.figsize"] = (7,4)

## Payoff and Minimax (coarse grid search)
Rows = Apprentice actions (4). Cols = Conductor actions (4). Values are payoff to Apprentice.

In [ ]:
M = np.array([
    [ 0.2, -0.1,  0.3, -0.2],
    [-0.1,  0.1, -0.2,  0.4],
    [ 0.3, -0.2,  0.2, -0.1],
    [-0.2,  0.4, -0.1,  0.1]
], dtype=float)

grid = np.linspace(0.0, 1.0, 51)

def simplex4():
    for a in grid:
        for b in grid:
            for c in grid:
                d = 1 - a - b - c
                if d >= 0: 
                    yield np.array([a,b,c,d], dtype=float)

# approximate minimax (Apprentice minimizes worst-case value)
Y = list(simplex4())
best_val = float("inf")
best_x = None
for x in simplex4():
    worst = max(float(x @ M @ y) for y in Y)
    if worst < best_val:
        best_val, best_x = worst, x

# Conductor best-response to best_x
vals = [float(best_x @ M @ y) for y in Y]
best_y = Y[int(np.argmin(vals))]
minimax_value = float(best_val)
minimax_value

## Repeated Game Rollout + Charts
We simulate 200 steps with small probability updates toward the sampled actions.

In [ ]:
T = 200
x = best_x.copy()
y = best_y.copy()
rewards = []

for t in range(T):
    ai = np.random.choice(4, p=x)
    cj = np.random.choice(4, p=y)
    rewards.append(float(M[ai, cj]))
    # light adaptive updates (keep distributions valid and smooth)
    x = 0.99 * x; x[ai] += 0.01; x = np.maximum(x, 1e-9); x /= x.sum()
    y = 0.99 * y; y[cj] += 0.01; y = np.maximum(y, 1e-9); y /= y.sum()

# Chart 1: Cumulative reward
plt.figure()
plt.plot(np.cumsum(rewards))
plt.title("Cumulative Reward (Apprentice)")
plt.xlabel("step"); plt.ylabel("sum r")
plt.tight_layout()
plt.savefig("img/disney_cumreward.png", dpi=160)
plt.show()

# Chart 2: joint strategy heatmap (outer product of final strategies)
heat = np.outer(x, y)
plt.figure()
plt.imshow(heat, aspect="auto")
plt.title("Joint Strategy Heatmap (final)")
plt.colorbar(); plt.xlabel("Conductor action"); plt.ylabel("Apprentice action")
plt.tight_layout()
plt.savefig("img/disney_heatmap.png", dpi=160)
plt.show()

## Metrics (copy these into README)
We export a small dictionary; you can paste the values into your README.

In [ ]:
metrics_disney = {
    "minimax_value_approx": minimax_value,
    "T": T,
    "final_strategy_A": x.round(4).tolist(),
    "final_strategy_C": y.round(4).tolist(),
    "cum_reward_last50_mean": float(np.mean(rewards[-50:]))
}
with open("metrics_disney.json", "w") as f:
    json.dump(metrics_disney, f, indent=2)
metrics_disney

## Sanity Checks
Light assertions to keep things reproducible and safe.

In [ ]:
assert np.isclose(x.sum(), 1.0) and np.all(x >= 0)
assert np.isclose(y.sum(), 1.0) and np.all(y >= 0)
assert M.shape == (4,4)
assert np.isfinite(np.sum(rewards))
print("Checks passed.")